In [63]:
from helper_client_setup import client, model

This method is not used

In [64]:
import requests

def get_weather(location):
    if not location or location.strip() == "":
        raise ValueError("Location cannot be empty")

    geourl = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&format=json"
    params = {
        "appid": "1234567890",
        "q": location,
        "units": "metric"
    }
    response = requests.get(geourl, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    return data



In [65]:
from datetime import datetime
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})


In [66]:
message = []

message.append(
    {
        "role": "user",
        "content": "What is the exact time formated as HH:MM:SS? what is your name?"
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=message,
    tools=[get_current_datetime_schema],
    tool_choice={"type": "auto"}
)

message.append({
    "role": "assistant",
    "content": response.content
})

message


[{'role': 'user',
  'content': 'What is the exact time formated as HH:MM:SS? what is your name?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ACBpUZ3BBnt4TmXX3qZWo3', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [67]:
# Execute the tool
tool_use = response.content[0]
result = get_current_datetime(**tool_use.input)
result

'10:57:21'

In [68]:
message.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": result,
            "is_error": False
        }
    ]
})

message

[{'role': 'user',
  'content': 'What is the exact time formated as HH:MM:SS? what is your name?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ACBpUZ3BBnt4TmXX3qZWo3', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01ACBpUZ3BBnt4TmXX3qZWo3',
    'content': '10:57:21',
    'is_error': False}]}]

In [69]:
final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=message,
    tools=[get_current_datetime_schema]
)
print(final_response)
print("--------------------------------")
print(final_response.content[0].text)

Message(id='msg_011CeAeSrVk9cGFAU8QWYtfv', container=None, content=[TextBlock(citations=None, text="The exact time formatted as HH:MM:SS is **10:57:21**.\n\nAs for my name, I'm **Claude**, an AI assistant made by Anthropic. How can I help you today?", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=704, output_tokens=49, server_tool_use=None, service_tier='standard'))
--------------------------------
The exact time formatted as HH:MM:SS is **10:57:21**.

As for my name, I'm **Claude**, an AI assistant made by Anthropic. How can I help you today?
